# 4.1 计算图优化

通过编译器优化计算图，减少不必要的计算和内存访问，最大化硬件利用率。核心优化包括算子融合、死代码消除、常量折叠和内存布局优化。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 算子融合（Operator Fusion）

将多个连续算子合并为单个算子执行，减少中间结果的内存读写。这是编译器优化中最重要的一项。

In [ ]:
class UnfusedLinearReLU(nn.Module):
    """未融合: Linear + ReLU 两个独立算子"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        h = self.linear(x)
        return self.relu(h)

class FusedLinearReLU(nn.Module):
    """融合: Linear + ReLU 合并为单个kernel"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        self.bias = nn.Parameter(torch.randn(out_dim))

    def forward(self, x):
        return F.linear(x, self.weight, self.bias).clamp(min=0)

in_dim, out_dim = 512, 256
unfused = UnfusedLinearReLU(in_dim, out_dim)
fused = FusedLinearReLU(in_dim, out_dim)
fused.weight.data.copy_(unfused.linear.weight.data)
fused.bias.data.copy_(unfused.linear.bias.data)

x = torch.randn(128, in_dim)

with torch.no_grad():
    out_unfused = unfused(x)
    out_fused = fused(x)
    diff = (out_unfused - out_fused).norm() / out_unfused.norm() * 100

n_iters = 1000
with torch.no_grad():
    start = time.time()
    for _ in range(n_iters):
        _ = unfused(x)
    unfused_time = time.time() - start

    start = time.time()
    for _ in range(n_iters):
        _ = fused(x)
    fused_time = time.time() - start

print(f"=== 算子融合: Linear + ReLU ===")
print(f"输出差异: {diff:.6f}% (数学等价)")
print(f"未融合时间: {unfused_time:.4f}s")
print(f"融合时间: {fused_time:.4f}s")
print(f"加速比: {unfused_time/fused_time:.2f}x")
print(f"\n融合节省: 1次中间张量的内存写入+读取")

class UnfusedLinearBNReLU(nn.Module):
    """未融合: Linear + BatchNorm + ReLU"""
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)
        self.bn = nn.BatchNorm1d(dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.bn(self.linear(x)))

class FusedLinearBNReLU(nn.Module):
    """融合: BN参数吸收到Linear中 + ReLU"""
    def __init__(self, linear, bn):
        super().__init__()
        with torch.no_grad():
            bn_var = bn.running_var
            bn_mean = bn.running_mean
            bn_weight = bn.weight
            bn_bias = bn.bias
            eps = bn.eps
            scale = bn_weight / torch.sqrt(bn_var + eps)
            self.weight = nn.Parameter(linear.weight * scale.unsqueeze(1))
            self.bias = nn.Parameter(linear.bias * scale + bn_bias - bn_mean * scale)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias).clamp(min=0)

dim = 256
unfused_lbr = UnfusedLinearBNReLU(dim)
unfused_lbr.eval()
fused_lbr = FusedLinearBNReLU(unfused_lbr.linear, unfused_lbr.bn)

x = torch.randn(64, dim)
with torch.no_grad():
    out_u = unfused_lbr(x)
    out_f = fused_lbr(x)
    diff = (out_u - out_f).norm() / out_u.norm() * 100

print(f"\n=== 算子融合: Linear + BatchNorm + ReLU ===")
print(f"输出差异: {diff:.6f}%")
print(f"BN参数吸收到Linear后: 3个算子 -> 1个算子")
print(f"节省: 2次中间张量读写 + BN计算")

### 常量折叠 & 死代码消除

In [ ]:
class ModelWithRedundancy(nn.Module):
    """包含冗余计算的模型"""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.fc3 = nn.Linear(dim, dim)
        self.constant_scale = nn.Parameter(torch.tensor(2.0), requires_grad=False)
        self.constant_bias = nn.Parameter(torch.tensor(0.5), requires_grad=False)

    def forward(self, x):
        h = self.fc1(x)
        dead_branch = self.fc2(x) * 0
        h = h * self.constant_scale + self.constant_bias
        h = F.relu(h)
        h = self.fc3(h)
        return h + dead_branch

class OptimizedModel(nn.Module):
    """优化后: 常量折叠 + 死代码消除"""
    def __init__(self, original):
        super().__init__()
        self.fc1 = nn.Linear(original.fc1.in_features, original.fc1.out_features)
        self.fc3 = nn.Linear(original.fc3.in_features, original.fc3.out_features)
        with torch.no_grad():
            self.fc1.weight.copy_(original.fc1.weight * original.constant_scale)
            self.fc1.bias.copy_(original.fc1.bias * original.constant_scale + original.constant_bias)
            self.fc3.weight.copy_(original.fc3.weight)
            self.fc3.bias.copy_(original.fc3.bias)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc3(h)

dim = 256
original = ModelWithRedundancy(dim)
optimized = OptimizedModel(original)

x = torch.randn(32, dim)
with torch.no_grad():
    out_orig = original(x)
    out_opt = optimized(x)
    diff = (out_orig - out_opt).norm() / out_orig.norm() * 100

orig_params = sum(p.numel() for p in original.parameters())
opt_params = sum(p.numel() for p in optimized.parameters())

n_iters = 500
with torch.no_grad():
    start = time.time()
    for _ in range(n_iters):
        _ = original(x)
    orig_time = time.time() - start

    start = time.time()
    for _ in range(n_iters):
        _ = optimized(x)
    opt_time = time.time() - start

print(f"=== 常量折叠 + 死代码消除 ===")
print(f"输出差异: {diff:.6f}%")
print(f"原始参数: {orig_params:,}")
print(f"优化后参数: {opt_params:,} (减少{(1-opt_params/orig_params)*100:.0f}%)")
print(f"原始时间: {orig_time:.4f}s")
print(f"优化后时间: {opt_time:.4f}s (加速{orig_time/opt_time:.2f}x)")
print(f"\n优化内容:")
print(f"1. 常量折叠: scale*bias预计算合并到fc1权重")
print(f"2. 死代码消除: fc2(x)*0 永远为0，移除")
print(f"3. 算子融合: fc1+scale+bias+relu 合并")

### PyTorch torch.compile 实践

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, dim=128, n_heads=4, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': nn.MultiheadAttention(dim, n_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)),
                'ln1': nn.LayerNorm(dim),
                'ln2': nn.LayerNorm(dim),
            }) for _ in range(n_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            h = layer['ln1'](x)
            h, _ = layer['attn'](h, h, h)
            x = x + h
            x = x + layer['ffn'](layer['ln2'](x))
        return x

model = SimpleTransformer(dim=128, n_heads=4, n_layers=4)
model.eval()
x = torch.randn(4, 32, 128)

with torch.no_grad():
    out_eager = model(x)

try:
    compiled_model = torch.compile(model, mode='reduce-overhead')
    with torch.no_grad():
        out_compiled = compiled_model(x)
    diff = (out_eager - out_compiled).norm() / out_eager.norm() * 100
    print(f"torch.compile编译成功")
    print(f"输出差异: {diff:.6f}%")

    n_iters = 100
    with torch.no_grad():
        start = time.time()
        for _ in range(n_iters):
            _ = model(x)
        eager_time = time.time() - start

        for _ in range(5):
            _ = compiled_model(x)
        start = time.time()
        for _ in range(n_iters):
            _ = compiled_model(x)
        compiled_time = time.time() - start

    print(f"Eager模式: {eager_time:.4f}s")
    print(f"Compiled模式: {compiled_time:.4f}s")
    print(f"加速比: {eager_time/compiled_time:.2f}x")
except Exception as e:
    print(f"torch.compile不可用: {e}")
    print(f"torch.compile需要PyTorch 2.0+")

print(f"\ntorch.compile优化内容:")
print(f"1. 自动算子融合")
print(f"2. 内存布局优化")
print(f"3. 减少Python开销")
print(f"4. 生成针对硬件优化的kernel")

### 产业级实战：torch.compile 多模式性能对比

PyTorch 2.0+ 的 `torch.compile` 是产业界最常用的图编译优化工具。它支持三种模式：
- `default`：平衡编译时间和运行时性能
- `reduce-overhead`：减少CUDA kernel launch开销（使用CUDAGraphs）
- `max-autotune`：尝试所有可能的kernel配置，选择最快的（编译最慢但运行最快）

In [ ]:
import time

class TransformerBlock(nn.Module):
    def __init__(self, dim=256, n_heads=4, ffn_dim=1024):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, dim),
        )
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

class SmallTransformer(nn.Module):
    def __init__(self, vocab_size=1000, dim=256, depth=4, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.Sequential(*[TransformerBlock(dim, n_heads) for _ in range(depth)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, input_ids):
        x = self.embed(input_ids)
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.head(x)

model = SmallTransformer(vocab_size=1000, dim=256, depth=4, n_heads=4)
model.eval()
dummy = torch.randint(0, 1000, (4, 64))

def benchmark_fn(fn, *args, warmup=5, runs=20):
    for _ in range(warmup):
        fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / runs * 1000
    return elapsed

eager_time = benchmark_fn(model, dummy)
print(f"Eager模式: {eager_time:.2f} ms")

modes = ['default', 'reduce-overhead', 'max-autotune']
results = {'eager': eager_time}
for mode in modes:
    try:
        compiled = torch.compile(model, mode=mode)
        with torch.no_grad():
            compiled_time = benchmark_fn(compiled, dummy)
        speedup = eager_time / compiled_time
        results[mode] = compiled_time
        print(f"torch.compile({mode}): {compiled_time:.2f} ms ({speedup:.2f}x)")
        del compiled
    except Exception as e:
        print(f"torch.compile({mode}): 失败 - {e}")

print(f"\n=== torch.compile 性能对比 ===")
print(f"{'模式':<20} {'延迟(ms)':<12} {'加速比':<10}")
print("-" * 42)
for name, t in results.items():
    speedup = f"{eager_time/t:.2f}x" if name != 'eager' else '1.00x'
    print(f"{name:<20} {t:<12.2f} {speedup:<10}")

print(f"\n产业界torch.compile使用建议:")
print(f"1. 开发阶段: eager模式 (调试方便)")
print(f"2. 部署阶段: reduce-overhead (GPU推理首选)")
print(f"3. 极致性能: max-autotune (编译慢但运行最快)")
print(f"4. 常见问题: 动态shape → 使用dynamic=True")
print(f"5. 调试: TORCH_LOGS='+dynamo' 查看编译日志")